# kbstatpy: your own analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimbostroem/kbstatpy/blob/master/analysis_template.ipynb)

A template for analysing **your own data** with
[kbstatpy](https://github.com/kimbostroem/kbstatpy) on Google Colab. Run the cells
from top to bottom:

1. **Setup** installs kbstatpy and its R packages (~1-2 min, once per session).
2. **Upload** your data file (`.csv` or `.xlsx`, one row per observation) and
   see its columns, or skip the upload and give a URL in step 3.
3. **Options**: fill in the file name and your column names.
4. **Run** fits the model and shows tables and figures.
5. **Download** the results as a zip file.

> **Files vanish with the runtime.** Uploads and results are deleted when the
> Colab session ends. To keep them, either **download the results** (step 5: one
> zip with all tables and figures, and `analysis.py`, the script that reruns the
> analysis), or mount your Google Drive (Drive icon in the Files pane on the
> left) and use paths such as `/content/drive/MyDrive/stats/my_data.csv`.
>
> **Whose notebook is this?** The moment you run or edit a cell, Colab keeps a
> private copy in your own Google account (*File → Save a copy in Drive*).

## 1. Setup

Installs kbstatpy and the R packages it relies on. Does nothing outside Colab,
where kbstatpy is expected to be installed already.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## 2. Upload your data

Run the cell and pick your file. It lands in `/content`, the notebook's working
directory, so the options below find it by its plain file name. The cell then
shows the columns and the first rows, which you need for the options.

Alternatively, drag the file into the Files pane (folder icon on the left).

**Data behind a link need no upload:** set `options.in_file` to the URL instead
of a file name. A sciebo/Nextcloud share link works too; add `/download` at its
end (`https://.../s/<token>/download`).

In [ ]:
import sys
import pandas as pd

if 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = list(files.upload())
else:
    uploaded = []
    print('Not on Colab: put your data file next to this notebook.')

for name in uploaded:
    data = pd.read_excel(name) if name.endswith(('.xlsx', '.xls')) else pd.read_csv(name)
    print(f'{name}: {len(data)} rows, columns: {", ".join(data.columns)}')
    display(data.head())

## 3. Options

Set `in_file` to the uploaded file's name, or to a URL (for a sciebo/Nextcloud
share the link ending in `/download`), and `y`, `x` and `id` to your column
names. Every commented-out line shows the value kbstatpy uses anyway: uncomment
a line only to change it, and delete what you do not need. Full reference:
[options table](https://github.com/kimbostroem/kbstatpy#options-reference) and
[statistical notes](https://github.com/kimbostroem/kbstatpy/blob/master/STATISTICAL_NOTES.md).

In [ ]:
from kbstatpy import Kbstat, KbstatOptions

options = KbstatOptions()

# ---------------------------------------------------------------------------
# Data in, results out                                              [required]
# ---------------------------------------------------------------------------
options.in_file = 'my_data.csv'             # the uploaded file's name, .csv or .xlsx, one row per observation
# options.in_file = 'https://uni-muenster.sciebo.de/s/<token>/download'   # or a URL / share link, no upload needed
options.out_dir = 'results/my_analysis'     # created if missing, in the working directory
# options.save_script = True                # DEFAULT: this code goes into out_dir with the results

# options.constraints = 'age > 18 & group != "pilot"'   # row filter, applied before everything else

# ---------------------------------------------------------------------------
# The model                                                         [required]
# ---------------------------------------------------------------------------
options.y  = 'my_outcome'                   # dependent variable, comma-separated for several
options.x  = 'group, condition'             # fixed factor(s), the first goes on the x-axis; '' = y ~ 1, the mean alone
options.id = 'subject'                      # random grouping factor, '' for no random effect

# options.interaction = 'auto'             # DEFAULT: every interaction the design supports
# options.interaction = ''                 # additive: main effects only, no interactions
# options.interaction = 'group, condition' # just these interact
# options.interaction = 'all'              # the full factorial; raises if not estimable
# options.interaction = 2                  # every factor in x, up to two-way
#
# 'auto' and 'all' differ only where a term is not estimable because cells
# are empty: 'auto' drops it and says so in Summary.txt, 'all' raises.

# options.covariate   = 'age'               # numeric covariates: in the model, out of the plots
# options.covariate   = 'age^2, log(dose)'  # curved: age + age^2 (centred), log of raw dose
# options.scale_covariates = False         # DEFAULT is True: centre/scale them (no test changes)
# options.slope       = 'condition'         # random slope(s) on options.id, not just an intercept

# Several grouping factors, comma-separated, are CROSSED. Use a slash to nest.
# An unsupported block term fits as a zero variance and a singular fit.
# options.id = 'subject, session'           # crossed: (1 | subject) + (1 | session)
# options.id = 'subject/repetition'         # nested:  (1 | subject) + (1 | subject:repetition)
# options.id = 'subject/session'            # several trials per session, session in x: needed
#                                           # against pseudo-replication (see STATISTICAL_NOTES)

# A formula is a complete alternative to the fields above, not an addition to
# them: y, x and id can all be left out. Write 'y' (or 'Y') on the left as a
# placeholder and options.y fills it in, one model per entry; write a real
# column name and that one is fitted, which is handy for looking at a single
# outcome without editing options.y.
# options.formula     = 'y ~ group * condition + (1 | subject)'      # each options.y
# options.formula     = 'score ~ group * condition + (1 | subject)'  # just this one

# ---------------------------------------------------------------------------
# Distribution, only when the outcome is not roughly normal
# ---------------------------------------------------------------------------
# options.distribution = 'normal'           # normal | gamma | binomial | poisson | inverse_gaussian
# options.link         = 'auto'             # DEFAULT: identity for normal, logit for binomial, log for the rest
# options.y_transform  = 'log(y)'           # transform instead of a GLMM; EMMs are back-transformed
# options.dispersion   = 'group'            # per-group dispersion for the glmmTMB families

# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------
# options.df_method        = 'auto'         # auto | kenward-roger | satterthwaite | asymptotic
# options.model_comparison = False          # AIC/BIC for additive vs two-way vs factorial
# options.test_value       = 175            # x = '': test the mean against this value (default 0)

# ---------------------------------------------------------------------------
# Post-hoc comparisons
# ---------------------------------------------------------------------------
# options.posthoc_compare    = 'auto'       # factor(s) to compare pairwise, '' or 'none' = off
# options.posthoc_correction = 'holm'       # holm | bonferroni | fdr | tukey | none
# options.posthoc_family     = 'cell'       # what the correction spans: cell | pooled | cross
# options.y_correction       = 'none'       # correction ACROSS several dependent variables
# options.split              = 'task'       # one model per level of this column (e.g. per task)
# options.split_correction   = 'FDR'        # each contrast corrected ACROSS the split levels

# A two-level factor gives one comparison per cell, and correcting one value does
# nothing, so pCorr equals p. Set posthoc_family='pooled' if the cells should be
# corrected against each other. Summary.txt states which family was used.

# ---------------------------------------------------------------------------
# Outliers
# ---------------------------------------------------------------------------
# options.remove_outliers_prefit  = False   # drop IQR outliers (1.5 x IQR per group) before fitting
# options.remove_outliers_postfit = False   # drop residual outliers (z > 3) and refit
# options.data_outliers = 'text'            # how excluded rows show in the plot: text | plot | none

# ---------------------------------------------------------------------------
# Labels and appearance
# ---------------------------------------------------------------------------
# options.title   = 'Study A'               # title prefix, 'none' suppresses it entirely
# options.y_units = 'Nm'                    # y-axis unit(s), comma-separated for multi-y
# options.x_units = ', mg'                  # unit(s) per x factor, by position; empty = no unit
# options.rename  = 'grp -> Group; grp: c1 -> Control'   # display names for variables and levels
# options.x_order = 'dose: low, medium, high'            # level order on the axis and in the tables

# options.plot_style      = 'auto'          # violin | bar | auto (bar for binary outcomes)
# options.y_scale         = 'linear'        # linear | log
# options.show_group_size = False           # annotate each group with its n
# options.show_emm_lines  = False           # reference line at each group's marginal mean
# options.color_scheme    = 'Set1'          # any matplotlib/seaborn palette name
# options.figure_display  = 'show_close'    # save_only | show_close | show_keep (all of them save)

# ---------------------------------------------------------------------------
# Optional extra analyses
# ---------------------------------------------------------------------------
# options.correlation         = 'a, b, c'   # pairwise correlations between numeric variables
# options.correlation_method  = 'pearson'   # pearson | spearman
# options.correlation_control = 'age'       # partial this out of every correlation
# options.profile_across      = 'dose'      # profile across one ordered factor (>= 3 levels)
# options.ordered             = 'dose'      # trend components (linear, quadratic, cubic) of ordered factors


## 4. Run

Fits the model, tests it, shows tables and figures, and writes everything to
`out_dir`. Rerun this cell after changing the options.

The options cell goes into `out_dir` too, as `analysis.py`: a script that
reruns exactly this analysis (`python3 analysis.py`, started from the folder
that holds the data file), so the results never get separated from the code
that made them.

In [ ]:
kb = Kbstat(options)
kb.run_save();

## 5. Download the results

Zips `out_dir`, results and `analysis.py`, and downloads it. The files are
also visible in the Files pane, where a right-click downloads a single one.

In [ ]:
import shutil, sys

archive = shutil.make_archive(options.out_dir.rstrip('/'), 'zip', options.out_dir)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)
else:
    print('Results zipped to', archive)